In [ ]:
# ========== COMPLETE AI CHATBOT TRAINER WITH MULTI-COLUMN SUPPORT ==========
import pandas as pd
import torch
import os
import io
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("AI CHATBOT TRAINER - STEP-BY-STEP MENU")
print("=" * 60)
print("Follow these steps in order:")
print("1. Install required libraries")
print("2. Upload your training CSV")
print("3. Prepare data for training")
print("4. Choose a pre-trained model")
print("5. Setup model and tokenizer")
print("6. Configure training")
print("7. Start training")
print("8. Load fine-tuned model")
print("9. Create chatbot")
print("10. Test with another CSV (optional)")
print("11. Start chatting")
print("12. Save to Google Drive (optional)")
print("13. Test accuracy (optional)")
print("=" * 60)

# Global variables with flags to track completion
df = None
question_cols = []
answer_cols = []
model_name = None
tokenizer = None
model = None
chatbot = None
save_path = "./my-fine-tuned-chatbot"
csv_filename = None
trainer = None
fine_tuned_model = None
fine_tuned_tokenizer = None
train_dataset = None
val_dataset = None
tokenized_train = None
tokenized_val = None

# Completion flags
steps_completed = {
    'step1': False,
    'step2': False,
    'step3': False,
    'step4': False,
    'step5': False,
    'step6': False,
    'step7': False,
    'step8': False,
    'step9': False
}

# ========== HELPER FUNCTIONS FOR COLUMN SELECTION ==========
def parse_column_input(input_str, max_cols):
    """Parse column input like '1,2,3' or '1-3' or '1 2 3'"""
    if not input_str or input_str.strip() == "":
        return []

    input_str = input_str.strip().replace(' ', ',')
    indices = []

    # Handle ranges like 1-3
    if '-' in input_str and ',' not in input_str:
        try:
            start, end = map(int, input_str.split('-'))
            if 1 <= start <= max_cols and 1 <= end <= max_cols and start <= end:
                indices = list(range(start, end + 1))
            else:
                print(f"Invalid range. Please use numbers between 1 and {max_cols}")
                return None
        except:
            print("Invalid range format. Use format: start-end (e.g., 1-3)")
            return None
    else:
        # Handle comma-separated list
        parts = input_str.replace(',', ' ').split()
        for part in parts:
            try:
                idx = int(part)
                if 1 <= idx <= max_cols:
                    indices.append(idx)
                else:
                    print(f"Invalid number: {part}. Please use numbers between 1 and {max_cols}")
                    return None
            except:
                print(f"Invalid input: {part}. Please enter numbers only")
                return None

    return sorted(list(set(indices)))  # Remove duplicates and sort

def display_column_selection(selected_cols, column_names, col_type):
    """Display currently selected columns"""
    if selected_cols:
        print(f"\nCurrent {col_type} columns selected:")
        for idx in selected_cols:
            print(f"   Column {idx}: {column_names[idx-1]}")
    else:
        print(f"\nNo {col_type} columns selected yet.")

# ========== STEP 1: INSTALL LIBRARIES ==========
def step1_install_libraries():
    global steps_completed
    print("\n" + "="*60)
    print("STEP 1: INSTALLING REQUIRED LIBRARIES")
    print("="*60)

    !pip install -q torch transformers datasets accelerate pandas numpy scikit-learn evaluate rouge-score

    print("Libraries installed successfully!")
    steps_completed['step1'] = True
    return True

# ========== STEP 2: UPLOAD TRAINING CSV ==========
def step2_upload_csv():
    global df, question_cols, answer_cols, csv_filename, steps_completed

    print("\n" + "="*60)
    print("STEP 2: UPLOAD TRAINING CSV")
    print("="*60)

    # Reset column selections
    question_cols = []
    answer_cols = []

    print("\nPlease enter the path to your training CSV file.")
    print("Examples:")
    print("  - /content/train.csv")
    print("  - /content/my_data.csv")
    print("  - /content/drive/MyDrive/dataset.csv")
    print("\nOr type 'upload' to upload a new file")
    print("-" * 60)

    file_input = input("Enter file path or 'upload': ").strip()

    if file_input.lower() == 'upload':
        from google.colab import files
        print("\nPlease select your CSV file...")
        uploaded = files.upload()

        if not uploaded:
            print("No file uploaded. Please try again.")
            return False

        csv_filename = list(uploaded.keys())[0]
        try:
            df = pd.read_csv(io.BytesIO(uploaded[csv_filename]), on_bad_lines='skip', engine='python')
            print(f"Uploaded: {csv_filename}")
        except Exception as e:
            print(f"Error reading CSV: {e}")
            return False
    else:
        csv_filename = file_input
        if os.path.exists(csv_filename):
            try:
                print(f"Loading {csv_filename}...")
                df = pd.read_csv(csv_filename, on_bad_lines='skip')
                print(f"Loaded: {csv_filename}")
            except Exception as e:
                print(f"Error loading file: {e}")
                return False
        else:
            print(f"File not found: {csv_filename}")
            return False

    if df is not None:
        print(f"\nDataset Info:")
        print(f"   Rows: {len(df)}")
        print(f"   Columns: {len(df.columns)}")

        print(f"\nColumns in dataset:")
        for i, col in enumerate(df.columns):
            print(f"   {i+1}. {col}")

        print("\n" + "="*60)
        print("MULTI-COLUMN SELECTION")
        print("="*60)
        print("You can select multiple columns for questions and answers.")
        print("Examples:")
        print("  - Single column: 1")
        print("  - Multiple columns: 1,2,3")
        print("  - Range: 1-3")
        print("  - Combination: 1,3-5,7")
        print("-" * 60)

        # Select question columns
        while True:
            display_column_selection(question_cols, df.columns, "QUESTION")
            q_input = input(f"\nSelect QUESTION columns (enter numbers, 'done' when finished, or 'clear' to reset): ").strip().lower()

            if q_input == 'done':
                if not question_cols:
                    print("You must select at least one question column.")
                    continue
                break
            elif q_input == 'clear':
                question_cols.clear()
                print("Question columns cleared.")
                continue
            elif q_input == '':
                print("Please enter column numbers.")
                continue

            new_indices = parse_column_input(q_input, len(df.columns))
            if new_indices is not None:
                question_cols.extend(new_indices)
                question_cols = sorted(list(set(question_cols)))
                print(f"Added columns: {[df.columns[i-1] for i in new_indices]}")

        # Select answer columns
        while True:
            display_column_selection(answer_cols, df.columns, "ANSWER")
            a_input = input(f"\nSelect ANSWER columns (enter numbers, 'done' when finished, or 'clear' to reset): ").strip().lower()

            if a_input == 'done':
                if not answer_cols:
                    print("You must select at least one answer column.")
                    continue
                break
            elif a_input == 'clear':
                answer_cols.clear()
                print("Answer columns cleared.")
                continue
            elif a_input == '':
                print("Please enter column numbers.")
                continue

            new_indices = parse_column_input(a_input, len(df.columns))
            if new_indices is not None:
                answer_cols.extend(new_indices)
                answer_cols = sorted(list(set(answer_cols)))
                print(f"Added columns: {[df.columns[i-1] for i in new_indices]}")

        question_column_names = [df.columns[i-1] for i in question_cols]
        answer_column_names = [df.columns[i-1] for i in answer_cols]

        print(f"\nFinal selection:")
        print(f"   Question columns: {question_column_names}")
        print(f"   Answer columns: {answer_column_names}")

        print("\nPreview (first 3 rows):")
        try:
            preview_cols = list(set(question_column_names + answer_column_names))
            preview = df[preview_cols].head(3)
            print(preview.to_string(index=False))
        except Exception as e:
            print(f"Could not show preview: {e}")

        if len(df) > 50000:
            print(f"\nLarge dataset detected: {len(df):,} rows")
            subset_choice = input("Use only a subset for faster training? (yes/no): ").strip().lower()
            if subset_choice in ['yes', 'y']:
                try:
                    subset_size = int(input(f"How many rows? (Recommended: 10000-50000): ").strip())
                    df = df.head(subset_size)
                    print(f"Using first {subset_size} rows")
                except:
                    df = df.head(20000)
                    print(f"Using first 20,000 rows")

        steps_completed['step2'] = True
        return True
    else:
        return False

# ========== STEP 3: PREPARE DATA ==========
def step3_prepare_data():
    global df, question_cols, answer_cols, train_dataset, val_dataset, steps_completed

    print("\n" + "="*60)
    print("STEP 3: PREPARING DATA FOR TRAINING")
    print("="*60)

    if df is None:
        print("No dataset loaded. Please run Step 2 first.")
        return False

    from sklearn.model_selection import train_test_split
    from datasets import Dataset

    # Get column names from indices
    question_column_names = [df.columns[i-1] for i in question_cols]
    answer_column_names = [df.columns[i-1] for i in answer_cols]

    print("Cleaning data...")
    # Create a copy with selected columns
    selected_cols = list(set(question_column_names + answer_column_names))
    df_clean = df[selected_cols].copy()
    df_clean = df_clean.dropna()

    # Convert all columns to string and clean
    for col in selected_cols:
        df_clean[col] = df_clean[col].astype(str).str.strip()
        # Remove empty strings
        df_clean = df_clean[df_clean[col].str.len() > 0]

    print(f"Cleaned dataset: {len(df_clean):,} rows")

    print("Formatting conversations with multiple columns...")
    conversations = []

    for _, row in df_clean.iterrows():
        # Build question part from multiple columns
        question_parts = []
        for col in question_column_names:
            if pd.notna(row[col]) and str(row[col]).strip():
                question_parts.append(str(row[col]).strip())

        # Build answer part from multiple columns
        answer_parts = []
        for col in answer_column_names:
            if pd.notna(row[col]) and str(row[col]).strip():
                answer_parts.append(str(row[col]).strip())

        # Only create conversation if we have both question and answer parts
        if question_parts and answer_parts:
            question_text = " ".join(question_parts)
            answer_text = " ".join(answer_parts)
            conversation = f"Human: {question_text}\nAssistant: {answer_text}"
            conversations.append({'text': conversation})

    if not conversations:
        print("No valid conversations created. Check your data.")
        return False

    data_df = pd.DataFrame(conversations)
    train_df, val_df = train_test_split(data_df, test_size=0.2, random_state=42)

    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    print(f"Data prepared:")
    print(f"   Training samples: {len(train_df):,}")
    print(f"   Validation samples: {len(val_df):,}")
    print(f"   Total conversations: {len(data_df):,}")

    print("\nSample conversation format:")
    if len(conversations) > 0:
        sample = conversations[0]['text']
        print(sample[:200] + "..." if len(sample) > 200 else sample)

    steps_completed['step3'] = True
    return True

# ========== STEP 4: CHOOSE MODEL ==========
def step4_choose_model():
    global model_name, steps_completed

    print("\n" + "="*60)
    print("STEP 4: CHOOSE PRE-TRAINED MODEL")
    print("="*60)
    print("Available models:")
    print("1. microsoft/DialoGPT-small (Fast, 117M params) - RECOMMENDED")
    print("2. google/flan-t5-base (Good for instructions, 250M params)")
    print("3. microsoft/phi-2 (Powerful, 2.7B params - needs more RAM)")
    print("-" * 60)

    choice = input("Enter choice (1-3, press Enter for 1): ").strip() or "1"

    if choice == "1":
        model_name = "microsoft/DialoGPT-small"
    elif choice == "2":
        model_name = "google/flan-t5-base"
    elif choice == "3":
        model_name = "microsoft/phi-2"
    else:
        model_name = "microsoft/DialoGPT-small"

    print(f"\nSelected model: {model_name}")

    if "DialoGPT" in model_name:
        print("   Fast training, good for conversation")
    elif "flan" in model_name:
        print("   Good for instruction following")
    elif "phi" in model_name:
        print("   Powerful but needs more memory")

    steps_completed['step4'] = True
    return True

# ========== STEP 5: SETUP MODEL AND TOKENIZER ==========
def step5_setup_model():
    global model_name, tokenizer, model, train_dataset, val_dataset, tokenized_train, tokenized_val, steps_completed

    print("\n" + "="*60)
    print("STEP 5: SETUP MODEL AND TOKENIZER")
    print("="*60)

    if model_name is None:
        print("No model selected. Please run Step 4 first.")
        return False

    if train_dataset is None:
        print("No training data prepared. Please run Step 3 first.")
        return False

    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"Loading tokenizer for {model_name}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        print("Tokenizer loaded")
    except Exception as e:
        print(f"Error loading tokenizer: {e}")
        return False

    print(f"Loading model...")
    try:
        # Auto-detect hardware
        use_cuda = torch.cuda.is_available()
        device = torch.device("cuda" if use_cuda else "cpu")

        print(f"\n{'='*60}")
        print("HARDWARE DETECTED:")
        print(f"{'='*60}")
        print(f"   Device: {device}")
        if use_cuda:
            print(f"   GPU: {torch.cuda.get_device_name(0)}")
            print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        else:
            print(f"   CPU Mode: Training will be slower")
        print(f"{'='*60}\n")

        # Load model based on hardware
        if 'phi-2' in model_name and use_cuda:
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map="auto"
            )
        else:
            model = AutoModelForCausalLM.from_pretrained(model_name)
            # Move model to appropriate device
            model = model.to(device)

        print(f"Model loaded on: {model.device}")
    except Exception as e:
        print(f"Error loading model: {e}")
        return False

    print("Tokenizing data...")
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=256
        )

    try:
        tokenized_train = train_dataset.map(tokenize_function, batched=True)
        tokenized_val = val_dataset.map(tokenize_function, batched=True)

        tokenized_train = tokenized_train.remove_columns(["text"])
        tokenized_val = tokenized_val.remove_columns(["text"])

        print("Tokenization complete!")
        print(f"   Training tokens: {len(tokenized_train)}")
        print(f"   Validation tokens: {len(tokenized_val)}")

        steps_completed['step5'] = True
        return True
    except Exception as e:
        print(f"Error during tokenization: {e}")
        return False

# ========== STEP 6: CONFIGURE TRAINING (FIXED WITH FP16 DISABLED) ==========
def step6_configure_training():
    global trainer, tokenized_train, tokenized_val, model, tokenizer, steps_completed

    print("\n" + "="*60)
    print("STEP 6: CONFIGURE TRAINING")
    print("="*60)

    from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

    # Auto-detect hardware
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")

    # Move model to the correct device if not already there
    if str(model.device) != str(device):
        print(f"Moving model from {model.device} to {device}...")
        model = model.to(device)

    # Print hardware info
    print(f"\n{'='*60}")
    print("HARDWARE DETECTED FOR TRAINING:")
    print(f"{'='*60}")
    print(f"   Device: {device}")
    if use_cuda:
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
        print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        print(f"   CPU Mode: Training will be slower")

    # Configure training based on hardware
    print(f"\n{'='*60}")
    print("CONFIGURING TRAINING FOR YOUR HARDWARE:")
    print(f"{'='*60}")

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

    # Base configuration
    training_config = {
        "output_dir": "./my-chatbot",
        "num_train_epochs": 3,
        "warmup_steps": 100,
        "weight_decay": 0.01,
        "logging_dir": "./logs",
        "logging_steps": 10,
        "eval_strategy": "steps",
        "eval_steps": 50,
        "save_strategy": "steps",
        "save_steps": 100,
        "load_best_model_at_end": True,
        "report_to": "none",
        "save_total_limit": 2,
        "dataloader_num_workers": 2 if use_cuda else 4,
        "remove_unused_columns": False,  # Added for stability
    }

    # Hardware-specific optimizations - FP16 DISABLED for DialoGPT compatibility
    if use_cuda:
        # GPU optimizations - FP16 disabled to prevent errors with DialoGPT
        training_config.update({
            "per_device_train_batch_size": 4,  # Safe batch size
            "per_device_eval_batch_size": 4,
            "fp16": False,  # CRITICAL: FP16 disabled to fix the error
            "bf16": False,  # Also disable bf16
            "gradient_accumulation_steps": 2,  # Accumulate for stability
            "optim": "adamw_torch",
            "torch_compile": False,
            "dataloader_pin_memory": True,
        })
        print(f"   ✓ GPU Mode: Using batch size 4 (FP16 disabled for DialoGPT compatibility)")
    else:
        # CPU optimizations
        training_config.update({
            "per_device_train_batch_size": 2,
            "per_device_eval_batch_size": 2,
            "fp16": False,
            "gradient_accumulation_steps": 4,
            "optim": "adamw_torch",
            "dataloader_pin_memory": False,
            "torch_compile": False,
        })
        print(f"   ✓ CPU Mode: Using batch size 2 with gradient accumulation")

    # Create training arguments
    training_args = TrainingArguments(**training_config)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
    )

    print("\nTraining configured:")
    print(f"   Epochs: {training_args.num_train_epochs}")
    print(f"   Batch size: {training_args.per_device_train_batch_size}")
    print(f"   Gradient accumulation: {training_args.gradient_accumulation_steps}")
    print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
    print(f"   Device: {'GPU' if use_cuda else 'CPU'}")
    print(f"   Mixed precision: Disabled (for DialoGPT compatibility)")
    print(f"   Logging every: {training_args.logging_steps} steps")
    print(f"   Saving every: {training_args.save_steps} steps")
    print(f"   Evaluating every: {training_args.eval_steps} steps")

    steps_completed['step6'] = True
    return True

# ========== STEP 7: START TRAINING ==========
def step7_start_training():
    global trainer, save_path, tokenized_train, tokenized_val, steps_completed

    print("\n" + "="*60)
    print("STEP 7: START TRAINING")
    print("="*60)

    if trainer is None:
        print("Trainer not configured. Please run Step 6 first.")
        return False

    print("WARNING: Training will take time and resources")
    print(f"Training samples: {len(tokenized_train):,}")
    print(f"Validation samples: {len(tokenized_val):,}")

    # Show estimated time based on hardware
    use_cuda = torch.cuda.is_available()
    if use_cuda:
        print(f"GPU detected - Training should take ~30-60 minutes")
    else:
        print(f"CPU detected - Training may take several hours")
        print(f"Consider switching to GPU runtime for faster training")

    confirm = input("\nStart training? This may take a while. (yes/no): ").strip().lower()
    if confirm not in ['yes', 'y', '1']:
        print("Training cancelled.")
        return False

    print("Starting training...")
    print("="*60)

    try:
        trainer.train()
        print("\nTraining complete!")

        trainer.save_model(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"Model saved to: {save_path}")

        steps_completed['step7'] = True
        return True
    except Exception as e:
        print(f"Training failed: {e}")
        print("\nTroubleshooting tips:")
        print("1. If you're on CPU, try switching to GPU runtime (Runtime → Change runtime type)")
        print("2. If you're on GPU, try reducing batch size further")
        print("3. Check your RAM/GPU memory usage")
        return False

# ========== STEP 8: LOAD FINE-TUNED MODEL ==========
def step8_load_model():
    global fine_tuned_model, fine_tuned_tokenizer, save_path, steps_completed

    print("\n" + "="*60)
    print("STEP 8: LOAD FINE-TUNED MODEL")
    print("="*60)

    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"Loading fine-tuned model from {save_path}...")

    try:
        if not os.path.exists(save_path):
            print(f"Model not found at {save_path}")
            print("Please run training first (Step 7).")
            return False

        fine_tuned_tokenizer = AutoTokenizer.from_pretrained(save_path)
        fine_tuned_model = AutoModelForCausalLM.from_pretrained(save_path)

        if fine_tuned_tokenizer.pad_token is None:
            fine_tuned_tokenizer.pad_token = fine_tuned_tokenizer.eos_token

        print("Fine-tuned model loaded successfully!")
        print(f"   Device: {fine_tuned_model.device}")

        steps_completed['step8'] = True
        return True
    except Exception as e:
        print(f"Error loading model: {e}")
        return False

# ========== STEP 9: CREATE CHATBOT ==========
def step9_create_chatbot():
    global chatbot, fine_tuned_model, fine_tuned_tokenizer, steps_completed

    print("\n" + "="*60)
    print("STEP 9: CREATE CHATBOT")
    print("="*60)

    if fine_tuned_model is None or fine_tuned_tokenizer is None:
        print("Fine-tuned model not loaded. Please run Step 8 first.")
        return False

    class SimpleChatbot:
        def __init__(self, model, tokenizer):
            self.model = model
            self.tokenizer = tokenizer
            self.model.eval()
            self.history = []

        def generate_response(self, user_input, max_length=150):
            if self.history:
                context = "\n".join(self.history[-4:])
                prompt = f"{context}\nHuman: {user_input}\nAssistant:"
            else:
                prompt = f"Human: {user_input}\nAssistant:"

            inputs = self.tokenizer.encode(prompt, return_tensors="pt").to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    inputs,
                    max_length=inputs.shape[1] + max_length,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            response = full_text.split("Assistant:")[-1].strip()

            self.history.append(f"Human: {user_input}")
            self.history.append(f"Assistant: {response}")

            if len(self.history) > 10:
                self.history = self.history[-10:]

            return response

        def chat(self):
            print("\n" + "="*60)
            print("CHATBOT READY!")
            print("="*60)
            print("Commands:")
            print("  Type 'quit' to exit")
            print("  Type 'clear' to clear history")
            print("  Type 'test' to test with a question")
            print("="*60)

            while True:
                try:
                    user_input = input("\nYou: ").strip()

                    if user_input.lower() in ['quit', 'exit', 'bye']:
                        print("\nChatbot: Goodbye!")
                        break
                    elif user_input.lower() == 'clear':
                        self.history = []
                        print("\nChatbot: History cleared!")
                        continue
                    elif user_input.lower() == 'test':
                        test_question = input("Enter test question: ").strip()
                        print(f"Chatbot: ", end="", flush=True)
                        response = self.generate_response(test_question)
                        print(response)
                        continue
                    elif not user_input:
                        continue

                    print("Chatbot: ", end="", flush=True)
                    response = self.generate_response(user_input)
                    print(response)

                except KeyboardInterrupt:
                    print("\n\nChatbot: Goodbye!")
                    break
                except Exception as e:
                    print(f"\nError: {str(e)}")
                    print("Please try again.")

    chatbot = SimpleChatbot(fine_tuned_model, fine_tuned_tokenizer)
    print("Chatbot created successfully!")
    steps_completed['step9'] = True
    return True

# ========== STEP 10: TEST WITH ANOTHER CSV ==========
def step10_test_with_csv():
    global chatbot

    print("\n" + "="*60)
    print("STEP 10: TEST WITH ANOTHER CSV FILE")
    print("="*60)

    if chatbot is None:
        print("Chatbot not created. Please run Step 9 first.")
        return False

    test_choice = input("Test with another CSV file? (yes/no): ").strip().lower()

    if test_choice not in ['yes', 'y', '1']:
        print("Skipping CSV testing.")
        return True

    from google.colab import files

    print("\nUpload your TEST CSV file...")
    uploaded_test = files.upload()

    if not uploaded_test:
        print("No file uploaded. Skipping test.")
        return True

    test_filename = list(uploaded_test.keys())[0]
    test_df = pd.read_csv(io.BytesIO(uploaded_test[test_filename]))

    print(f"Test file loaded: {test_filename}")
    print(f"Test samples: {len(test_df)}")

    print("\n" + "="*60)
    print("SELECT TEST COLUMNS")
    print("="*60)
    print("Columns in test file:")
    for i, col in enumerate(test_df.columns):
        print(f"   {i+1}. {col}")

    print("\nSelect question columns (can be multiple):")
    test_question_cols = []
    while True:
        q_input = input("Enter question column numbers (e.g., 1,2,3 or 'done' when finished): ").strip().lower()

        if q_input == 'done':
            if not test_question_cols:
                print("You must select at least one question column.")
                continue
            break

        new_indices = parse_column_input(q_input, len(test_df.columns))
        if new_indices is not None:
            test_question_cols.extend([test_df.columns[i-1] for i in new_indices])
            test_question_cols = list(set(test_question_cols))
            print(f"Current question columns: {test_question_cols}")

    print(f"\nTesting with question columns: {test_question_cols}")
    test_questions = test_df.head(20)

    results = []
    for idx, row in test_questions.iterrows():
        question_parts = []
        for col in test_question_cols:
            if col in row and pd.notna(row[col]) and str(row[col]).strip():
                question_parts.append(str(row[col]).strip())

        if question_parts:
            question = " ".join(question_parts)
            print(f"\n[{idx+1}] Question: {question[:80]}...")
            try:
                response = chatbot.generate_response(question)
                print(f"    Response: {response[:100]}...")
                results.append({'question': question, 'response': response})
            except Exception as e:
                print(f"    Error: {e}")
                results.append({'question': question, 'error': str(e)})

    results_df = pd.DataFrame(results)
    results_filename = "test_results.csv"
    results_df.to_csv(results_filename, index=False)
    print(f"\nTest results saved to: {results_filename}")

    return True

# ========== STEP 11: START CHATTING ==========
def step11_start_chatting():
    global chatbot

    print("\n" + "="*60)
    print("STEP 11: START CHATTING")
    print("="*60)

    if chatbot is None:
        print("Chatbot not created. Please run previous steps.")
        return False

    print("Starting interactive chat session...")
    chatbot.chat()
    return True

# ========== STEP 12: SAVE TO GOOGLE DRIVE ==========
def step12_save_to_drive():
    global save_path, csv_filename, question_cols, answer_cols, model_name, df

    print("\n" + "="*60)
    print("STEP 12: SAVE TO GOOGLE DRIVE")
    print("="*60)

    if not os.path.exists(save_path):
        print(f"No model found at {save_path}. Please train a model first.")
        return False

    choice = input("Save model to Google Drive? (yes/no): ").strip().lower()

    if choice not in ['yes', 'y', '1']:
        print("Model remains in Colab only.")
        return True

    from google.colab import drive
    import shutil
    import datetime

    print("Mounting Google Drive...")
    drive.mount('/content/drive')

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    drive_path = f"/content/drive/MyDrive/chatbot-{timestamp}"

    os.makedirs(drive_path, exist_ok=True)

    print(f"Saving to: {drive_path}")
    try:
        for item in os.listdir(save_path):
            src = os.path.join(save_path, item)
            dst = os.path.join(drive_path, item)
            if os.path.isdir(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src, dst)
        print("Model files copied successfully!")
    except Exception as e:
        print(f"Could not copy all files: {e}")

    # Get column names from indices if df exists
    question_column_names = [df.columns[i-1] for i in question_cols] if df is not None else []
    answer_column_names = [df.columns[i-1] for i in answer_cols] if df is not None else []

    info = {
        'trained_on': timestamp,
        'original_file': csv_filename if csv_filename else "unknown",
        'question_columns': question_column_names,
        'answer_columns': answer_column_names,
        'model_used': model_name if model_name else "unknown"
    }

    import json
    info_path = os.path.join(drive_path, "info.json")
    with open(info_path, 'w') as f:
        json.dump(info, f, indent=2)

    print(f"Model saved to Google Drive!")
    print(f"Path: {drive_path}")

    return True

# ========== STEP 13: TEST ACCURACY ==========
def step13_test_accuracy():
    global chatbot, val_dataset

    print("\n" + "="*60)
    print("STEP 13: TEST ACCURACY")
    print("="*60)

    if chatbot is None:
        print("Chatbot not created. Please run Step 9 first.")
        return False

    print("\nAccuracy testing options:")
    print("1. Test with validation data (from training)")
    print("2. Test with new CSV file")
    print("3. Manual test with sample questions")
    print("-" * 60)

    choice = input("Enter choice (1-3): ").strip()

    if choice == "1":
        if val_dataset is None:
            print("No validation data available. Please run Step 3 first.")
            return False

        print(f"\nTesting with validation data ({len(val_dataset)} samples)...")
        print("This will take some time...")

        test_samples = min(100, len(val_dataset))
        print(f"Testing on {test_samples} samples...")

        from evaluate import load
        rouge = load("rouge")

        predictions = []
        references = []

        for i in range(test_samples):
            try:
                sample_text = val_dataset[i]["text"]
                question = sample_text.split("Assistant:")[0].replace("Human:", "").strip()
                expected_answer = sample_text.split("Assistant:")[1].strip() if "Assistant:" in sample_text else ""

                generated_answer = chatbot.generate_response(question)

                predictions.append(generated_answer)
                references.append(expected_answer)

                if i % 10 == 0:
                    print(f"Processed {i+1}/{test_samples} samples...")

            except Exception as e:
                print(f"Error processing sample {i}: {e}")
                continue

        if predictions and references:
            results = rouge.compute(predictions=predictions, references=references)
            print("\n" + "="*60)
            print("ACCURACY TEST RESULTS")
            print("="*60)
            print(f"Tested on: {len(predictions)} samples")
            print(f"ROUGE-1: {results['rouge1']:.4f}")
            print(f"ROUGE-2: {results['rouge2']:.4f}")
            print(f"ROUGE-L: {results['rougeL']:.4f}")
            print(f"ROUGE-Lsum: {results['rougeLsum']:.4f}")

    elif choice == "2":
        print("\nTesting with new CSV file...")
        from google.colab import files

        print("Upload your test CSV file...")
        uploaded = files.upload()

        if not uploaded:
            print("No file uploaded.")
            return False

        test_filename = list(uploaded.keys())[0]
        test_df = pd.read_csv(io.BytesIO(uploaded[test_filename]))

        print(f"Test file loaded: {test_filename}")
        print(f"Samples: {len(test_df)}")

        print("\nColumns in test file:")
        for i, col in enumerate(test_df.columns):
            print(f"   {i+1}. {col}")

        print("\nSelect question columns:")
        question_cols_test = []
        while True:
            q_input = input("Enter question column numbers (e.g., 1,2,3 or 'done'): ").strip().lower()
            if q_input == 'done':
                if not question_cols_test:
                    print("Select at least one column.")
                    continue
                break
            new_indices = parse_column_input(q_input, len(test_df.columns))
            if new_indices:
                question_cols_test.extend([test_df.columns[i-1] for i in new_indices])
                question_cols_test = list(set(question_cols_test))

        print("\nSelect answer columns:")
        answer_cols_test = []
        while True:
            a_input = input("Enter answer column numbers (e.g., 1,2,3 or 'done'): ").strip().lower()
            if a_input == 'done':
                if not answer_cols_test:
                    print("Select at least one column.")
                    continue
                break
            new_indices = parse_column_input(a_input, len(test_df.columns))
            if new_indices:
                answer_cols_test.extend([test_df.columns[i-1] for i in new_indices])
                answer_cols_test = list(set(answer_cols_test))

        # Continue with testing...
        print("Testing functionality to be implemented...")

    elif choice == "3":
        print("\nManual testing to be implemented...")

    return True

# ========== RUN ESSENTIAL STEPS WITH CHECKPOINTS ==========
def run_essential_steps():
    """Run essential steps only if not already completed"""
    print("\n" + "="*60)
    print("RUNNING ESSENTIAL STEPS")
    print("="*60)

    essential_steps = [
        ("1. Install libraries", step1_install_libraries, 'step1'),
        ("2. Upload training CSV", step2_upload_csv, 'step2'),
        ("3. Prepare data", step3_prepare_data, 'step3'),
        ("4. Choose model", step4_choose_model, 'step4'),
        ("5. Setup model and tokenizer", step5_setup_model, 'step5'),
        ("6. Configure training", step6_configure_training, 'step6'),
    ]

    for step_name, step_func, step_key in essential_steps:
        if steps_completed.get(step_key, False):
            print(f"\n{step_name} already completed. Skipping...")
            continue

        print(f"\n{'='*60}")
        print(f"RUNNING: {step_name}")
        print(f"{'='*60}")

        try:
            result = step_func()
            if result:
                print(f"{step_name} completed!")
            else:
                print(f"{step_name} failed!")
                print("Please fix the issue and try again.")
                return False
        except Exception as e:
            print(f"Error in {step_name}: {e}")
            print("Please fix the issue and try again.")
            return False

    print("\n" + "="*60)
    print("ESSENTIAL STEPS COMPLETED!")
    print("="*60)
    return True

# ========== SHOW MAIN MENU ==========
def show_main_menu():
    """Show the main menu after essential steps"""
    print("\n" + "="*60)
    print("CHATBOT TRAINING - MAIN MENU")
    print("="*60)
    print("Choose next step:")
    print("1. Start training the model")
    print("2. Load fine-tuned model")
    print("3. Create chatbot")
    print("4. Test with another CSV")
    print("5. Start chatting")
    print("6. Save to Google Drive")
    print("7. Test accuracy")
    print("8. Restart from beginning")
    print("9. Exit")
    print("="*60)

    while True:
        choice = input("\nEnter choice (1-9): ").strip()

        if choice == "1":
            if steps_completed.get('step6', False):
                step7_start_training()
            else:
                print("Please complete essential steps first (choose option 1 from main menu).")

        elif choice == "2":
            step8_load_model()

        elif choice == "3":
            if steps_completed.get('step8', False):
                step9_create_chatbot()
            else:
                print("Please load a fine-tuned model first (Step 2).")

        elif choice == "4":
            step10_test_with_csv()

        elif choice == "5":
            if steps_completed.get('step9', False):
                step11_start_chatting()
            else:
                print("Please create a chatbot first (Step 3).")

        elif choice == "6":
            step12_save_to_drive()

        elif choice == "7":
            step13_test_accuracy()

        elif choice == "8":
            # Reset everything
            global df, question_cols, answer_cols, model_name, tokenizer, model
            global chatbot, csv_filename, trainer, fine_tuned_model, fine_tuned_tokenizer
            global train_dataset, val_dataset, tokenized_train, tokenized_val

            df = None
            question_cols = []
            answer_cols = []
            model_name = None
            tokenizer = None
            model = None
            chatbot = None
            csv_filename = None
            trainer = None
            fine_tuned_model = None
            fine_tuned_tokenizer = None
            train_dataset = None
            val_dataset = None
            tokenized_train = None
            tokenized_val = None

            # Reset completion flags
            for key in steps_completed:
                steps_completed[key] = False

            print("\nSystem reset! Starting fresh...")
            return "restart"

        elif choice == "9":
            print("\nThank you for using AI Chatbot Trainer! Goodbye!")
            return "exit"

        else:
            print("Invalid choice. Please enter 1-9.")

# ========== START PROGRAM ==========
def start_program():
    """Main entry point"""
    print("\n" + "="*60)
    print("WELCOME TO AI CHATBOT TRAINER")
    print("="*60)

    while True:
        print("\n" + "="*60)
        print("MAIN MENU")
        print("="*60)
        print("Choose execution mode:")
        print("1. Run essential steps (Steps 1-6)")
        print("2. Start training, Load model, Create chatbot")
        print("3. Test with another CSV")
        print("4. Start chatting")
        print("5. Save to Google Drive")
        print("6. Test accuracy")
        print("7. Exit")
        print("="*60)

        mode_choice = input("\nEnter choice (1-7): ").strip()

        if mode_choice == "1":
            if run_essential_steps():
                print("\nEssential steps completed! Showing main menu...")
                while True:
                    result = show_main_menu()
                    if result == "exit":
                        return
                    elif result == "restart":
                        break

        elif mode_choice == "2":
            print("\n" + "="*60)
            print("STARTING COMPLETE TRAINING PIPELINE")
            print("="*60)

            # Check if essential steps are completed
            if not all([steps_completed.get('step1', False),
                       steps_completed.get('step2', False),
                       steps_completed.get('step3', False),
                       steps_completed.get('step4', False),
                       steps_completed.get('step5', False),
                       steps_completed.get('step6', False)]):
                print("\nEssential steps not completed yet.")
                print("Running essential steps first...")
                if not run_essential_steps():
                    print("Essential steps failed. Cannot continue.")
                    continue

            # Run training pipeline
            steps_to_run = [
                ("Start training", step7_start_training, 'step7'),
                ("Load fine-tuned model", step8_load_model, 'step8'),
                ("Create chatbot", step9_create_chatbot, 'step9'),
            ]

            for step_name, step_func, step_key in steps_to_run:
                if steps_completed.get(step_key, False):
                    print(f"\n{step_name} already completed. Skipping...")
                    continue

                print(f"\n{'='*60}")
                print(f"RUNNING: {step_name}")
                print(f"{'='*60}")

                try:
                    result = step_func()
                    if result:
                        print(f"{step_name} completed!")
                    else:
                        print(f"{step_name} failed!")
                        break
                except Exception as e:
                    print(f"Error in {step_name}: {e}")
                    break

            print("\nComplete training pipeline finished!")
            print("You can now start chatting or test accuracy.")

        elif mode_choice == "3":
            step10_test_with_csv()

        elif mode_choice == "4":
            if steps_completed.get('step9', False):
                step11_start_chatting()
            else:
                print("\nChatbot not created yet.")
                print("Please create a chatbot first (choose option 1 or 2).")

        elif mode_choice == "5":
            step12_save_to_drive()

        elif mode_choice == "6":
            step13_test_accuracy()

        elif mode_choice == "7":
            print("\nThank you for using AI Chatbot Trainer! Goodbye!")
            break

        else:
            print("Invalid choice. Please enter 1-7.")

# ========== START PROGRAM ==========
if __name__ == "__main__":
    start_program()


AI CHATBOT TRAINER - STEP-BY-STEP MENU
Follow these steps in order:
1. Install required libraries
2. Upload your training CSV
3. Prepare data for training
4. Choose a pre-trained model
5. Setup model and tokenizer
6. Configure training
7. Start training
8. Load fine-tuned model
9. Create chatbot
10. Test with another CSV (optional)
11. Start chatting
12. Save to Google Drive (optional)
13. Test accuracy (optional)

WELCOME TO AI CHATBOT TRAINER

MAIN MENU
Choose execution mode:
1. Run essential steps (Steps 1-6)
2. Start training, Load model, Create chatbot
3. Test with another CSV
4. Start chatting
5. Save to Google Drive
6. Test accuracy
7. Exit

Enter choice (1-7): 1

RUNNING ESSENTIAL STEPS

RUNNING: 1. Install libraries

STEP 1: INSTALLING REQUIRED LIBRARIES
Libraries installed successfully!
1. Install libraries completed!

RUNNING: 2. Upload training CSV

STEP 2: UPLOAD TRAINING CSV

Please enter the path to your training CSV file.
Examples:
  - /content/train.csv
  - /content/my

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cuda:0
Tokenizing data...


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

...

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Fine-tuned model loaded successfully!
   Device: cpu

Enter choice (1-9): 3

STEP 9: CREATE CHATBOT
Chatbot created successfully!

Enter choice (1-9): 5

STEP 11: START CHATTING
Starting interactive chat session...

CHATBOT READY!
Commands:
  Type 'quit' to exit
  Type 'clear' to clear history
  Type 'test' to test with a question

You: What are the common symptoms and treatment options for [Specific Condition mentioned in the data]?
Chatbot: 

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Error: probability tensor contains either `inf`, `nan` or element < 0
Please try again.

You: Given a patient with [Symptom A] and [Symptom B], what preliminary questions should I ask to clarify their condition?
Chatbot: 
Error: probability tensor contains either `inf`, `nan` or element < 0
Please try again.

You: Can you explain the long-term management for this condition? Also, what are the potential side effects of the primary medication used?
Chatbot: 
Error: probability tensor contains either `inf`, `nan` or element < 0
Please try again.


Chatbot: Goodbye!


KeyboardInterrupt: Interrupted by user